# Proyecto Final de Inteligencia Artificial
## SkinGuard: Asistente Dermatológico Inteligente para Ejecución Offline

**Curso:** Inteligencia Artificial / Procesamiento de Lenguaje Natural (PLN)  
**Integrantes:**
* Andrés Felipe Bernal (Cód. 7003748)
* Andres Camilo Bernal (Cód. 7003932)
* Filocaris Triana (Cód. 7003936)
* Santiago Gayón (Cód. 7003880)

> **Objetivo:** Adaptar y refinar un modelo de lenguaje de tamaño reducido (`microsoft/Phi-3-mini-4k-instruct`) utilizando la técnica **LoRA (PEFT)** sobre el dataset especializado **SkinGuard** y documentar el flujo de trabajo completo para exportar este modelo entrenado a formatos ligeros (**ONNX** y **GGUF**) y aplicar **cuantización a 4-bits**, habilitando su despliegue y ejecución offline directamente en smartphones (Android/iOS).


## 1. Configuración de Entorno e Instalación de Dependencias
Para ejecutar este proyecto en Google Colab con una GPU T4, primero instalamos el ecosistema de Hugging Face y las librerías necesarias para el Fine-Tuning y la conversión a formatos optimizados.


In [4]:
# Instalar dependencias necesarias para Fine-Tuning y Conversión (con versiones estables para Colab)
!pip install -q "transformers==4.41.0" "datasets==2.19.0" "peft==0.10.0" "trl==0.8.6" "accelerate==0.30.0" bitsandbytes "optimum[onnxruntime]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 46.2 MB/s eta 0:00:00


## 2. Inicialización del Modelo Base y Tokenizador (Cuantización en 4-bits)
Usamos la cuantización NF4 en 4-bits para cargar el modelo `microsoft/Phi-3-mini-4k-instruct` reduciendo el consumo de VRAM a menos de 4GB, lo cual permite realizar el entrenamiento en la GPU T4 gratuita de Colab.


In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Configuración de cuantización NF4 (4-bits)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model_id = "microsoft/Phi-3-mini-4k-instruct"

# Cargar modelo base
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    trust_remote_code=True,
    device_map="auto"
)
model.config.use_cache = False

# Cargar tokenizador
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Modelo base y tokenizador cargados con éxito en 4-bits.")
print("Dispositivo asignado:", next(model.parameters()).device)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_phi3.py: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Modelo base y tokenizador cargados con éxito en 4-bits.
Dispositivo asignado: cuda:0


## 3. Carga y Procesamiento del Dataset SkinGuard
Cargamos el dataset `Dataset_SkinGuard.jsonl` que contiene consultas dermatológicas y respuestas recomendadas estructuradas en formato de mensajes. Convertimos el diálogo al formato de chat específico de `Phi-3-mini` empleando el template del tokenizador.


In [6]:
import json
import os
from datasets import Dataset

# Ruta del dataset en Colab (asegúrate de subir Dataset_SkinGuard.jsonl)
FILE_PATH = "Dataset_SkinGuard.jsonl"

if not os.path.exists(FILE_PATH):
    print(f"Buscando archivo en el directorio del notebook o subdirectorios...")
    # Buscar de forma adaptativa
    if os.path.exists("Proyecto IA/Dataset_SkinGuard.jsonl"):
        FILE_PATH = "Proyecto IA/Dataset_SkinGuard.jsonl"
    elif os.path.exists("../Proyecto IA/Dataset_SkinGuard.jsonl"):
        FILE_PATH = "../Proyecto IA/Dataset_SkinGuard.jsonl"
    elif os.path.exists("converted_dataset.jsonl"):
        FILE_PATH = "converted_dataset.jsonl"
    elif os.path.exists("../converted_dataset.jsonl"):
        FILE_PATH = "../converted_dataset.jsonl"
    else:
        raise FileNotFoundError(f"No se encontró 'Dataset_SkinGuard.jsonl' o 'converted_dataset.jsonl'. Por favor, súbelo a tu sesión de Colab.")

raw_data = []
with open(FILE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            raw_data.append(json.loads(line))

# Aplicar Chat Template del tokenizador de Phi-3
def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

# Crear dataset de Hugging Face y mapearlo
dataset = Dataset.from_list(raw_data)
dataset = dataset.map(format_chat, remove_columns=["messages"])

print(f"Dataset cargado correctamente. Número de ejemplos: {len(dataset)}")
print("\n--- Ejemplo de diálogo formateado para entrenamiento ---")
print(dataset[0]["text"])


Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Dataset cargado correctamente. Número de ejemplos: 50

--- Ejemplo de diálogo formateado para entrenamiento ---
<|user|>
Tengo un lunar oscuro que ha aumentado de tamaño y presenta bordes irregulares.<|end|>
<|assistant|>
La lesión descrita presenta signos clínicos sospechosos para melanoma cutáneo. Se recomienda evaluación dermatoscópica urgente y biopsia excisional.<|end|>
<|endoftext|>


## 4. Evaluación del Modelo Base (Antes de Entrenamiento)
Evaluamos cómo responde el modelo base `Phi-3-mini-4k-instruct` frente a consultas dermatológicas típicas del dataset para tener una referencia cualitativa.


In [7]:
# Consultas de prueba representativas
test_prompts = [
    "Tengo un lunar oscuro que ha aumentado de tamaño y presenta bordes irregulares.",
    "Tengo placas rojas con descamación en los codos y rodillas.",
    "Tengo picazón intensa y resequedad en las manos."
]

def evaluate_model(model, tokenizer, prompts, title=""):
    print(f"=== {title} ===")
    for prompt in prompts:
        messages = [{"role": "user", "content": prompt}]
        input_ids = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                input_ids,
                max_new_tokens=150,
                do_sample=True,
                temperature=0.7,
                top_k=50,
                top_p=0.95
            )

        decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
        # Extraer solo la respuesta del asistente
        assistant_response = decoded.split("assistant\n")[-1].strip()
        print(f"Pregunta: {prompt}")
        print(f"Respuesta:\n{assistant_response}")
        print("-" * 50)

evaluate_model(model, tokenizer, test_prompts, "Respuestas del Modelo Base")


=== Respuestas del Modelo Base ===


Pregunta: Tengo un lunar oscuro que ha aumentado de tamaño y presenta bordes irregulares.
Respuesta:
Tengo un lunar oscuro que ha aumentado de tamaño y presenta bordes irregulares. Esta descripción sugiere que podrías estar hablando de una luna que ha experimentado un evento de lava lunar, donde la superficie de la luna se ha vuelto más oscura debido a la acumulación de materiales de lava. Los bordes irregulares pueden ser causados por la forma en que se solidificó la lava o por impactos que han alterado la superficie.


Para investigar más sobre este fenómeno, se debería:

1. Realizar imágenes satelitales de alta resolución para documentar las características de la luna y la distribución de la lava.

2.
--------------------------------------------------
Pregunta: Tengo placas rojas con descamación en los codos y rodillas.
Respuesta:
Tengo placas rojas con descamación en los codos y rodillas. La descripción de "placas rojas con descamación en los codos y rodillas" sugiere la presencia 

## 5. Configuración de LoRA y Entrenamiento SFT (Supervised Fine-Tuning)
Configuramos los adaptadores LoRA para aplicarse sobre las matrices de atención (`q_proj`, `k_proj`, `v_proj`, `o_proj`) y las capas de proyección intermedia de la red neuronal. Luego configuramos y ejecutamos el entrenamiento.


In [8]:
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments
from trl import SFTTrainer

# Configuración de LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

# Enlazar adaptadores al modelo base
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Argumentos de entrenamiento para optimización en Colab T4
training_args = TrainingArguments(
    output_dir="./skinguard-finetuned",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    num_train_epochs=8,           # Entrenar durante 8 épocas para robustez en el dominio
    logging_steps=5,
    fp16=True,
    save_strategy="epoch",
    report_to="none",             # Evitar loguearse en Weights & Biases
)

# Configurar el entrenador SFT (Supervised Fine-Tuning)
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=1024,
    tokenizer=tokenizer,
    packing=False,
)

print("Entrenador configurado correctamente. Iniciando Fine-Tuning...")


trainable params: 8,912,896 || all params: 3,829,992,448 || trainable%: 0.23271314815918928


Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Entrenador configurado correctamente. Iniciando Fine-Tuning...


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [9]:
# Iniciar entrenamiento
trainer.train()
print("Entrenamiento completado.")


Step,Training Loss
5,2.717800
10,1.818400
15,1.342400
20,1.181900
25,1.086500
30,0.891500
35,0.887600
40,0.822100
45,0.690600
50,0.721400


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in ver

Entrenamiento completado.


## 6. Evaluación del Modelo Afinado (Después de Entrenamiento)
Evaluamos el comportamiento del modelo afinado usando exactamente los mismos prompts que en la evaluación base para observar los cambios en el tono, diagnóstico y tratamiento específico según el dataset de SkinGuard.


In [10]:
# Limpiar caché de GPU
torch.cuda.empty_cache()

# Probar modelo afinado
evaluate_model(model, tokenizer, test_prompts, "Respuestas del Modelo SkinGuard Afinado")


=== Respuestas del Modelo SkinGuard Afinado ===
Pregunta: Tengo un lunar oscuro que ha aumentado de tamaño y presenta bordes irregulares.
Respuesta:
Tengo un lunar oscuro que ha aumentado de tamaño y presenta bordes irregulares. El cuadro es compatible con carcinoma basocelular.
--------------------------------------------------
Pregunta: Tengo placas rojas con descamación en los codos y rodillas.
Respuesta:
Tengo placas rojas con descamación en los codos y rodillas. La clínica es compatible con psoriasis en placas.
--------------------------------------------------
Pregunta: Tengo picazón intensa y resequedad en las manos.
Respuesta:
Tengo picazón intensa y resequedad en las manos. La sintomatología sugiere dermatitis de contacto o dermatitis atópica. Debe descartarse una alergia al cosmético o al producto de higiene.
--------------------------------------------------


In [11]:
# Guardar los adaptadores LoRA entrenados
adapters_dir = "./skinguard-adapters"
model.save_pretrained(adapters_dir)
tokenizer.save_pretrained(adapters_dir)
print(f"Adaptadores LoRA guardados en {adapters_dir}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Adaptadores LoRA guardados en ./skinguard-adapters


## 7. Fusión de Pesos y Exportación para Aplicación Móvil Offline
Para utilizar el asistente en una aplicación móvil sin conexión a internet (totalmente offline), no podemos usar la configuración cuantizada en 4-bits más adaptadores en tiempo de ejecución. En su lugar, requerimos:
1. **Fusionar (Merge)** los pesos de los adaptadores LoRA entrenados en el modelo base original en precisión completa (FP16).
2. **Exportar** el modelo fusionado a formatos compatibles con móviles (**ONNX** o **GGUF**).
3. **Cuantizar** el modelo resultante (a 4-bits) para que quepa en la memoria RAM y almacenamiento de un dispositivo móvil (reduciendo su tamaño de ~7.6 GB a ~2.2 GB).


### Paso 7.1: Fusión (Merge) de adaptadores con el Modelo Base
Cargamos el modelo base en precisión de 16 bits (float16) sin cuantización de bitsandbytes (ya que los pesos cuantizados en 4-bits no se pueden fusionar matemáticamente de forma directa con LoRA). Luego cargamos los adaptadores guardados y realizamos la fusión.


In [12]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import gc

# Liberar memoria de GPU
try:
    del model
    del trainer
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

print("Cargando modelo base en precisión FP16...")
model_id = "microsoft/Phi-3-mini-4k-instruct"
adapters_dir = "./skinguard-adapters"
merged_output_dir = "./skinguard-merged"

# Cargar modelo base en FP16 en la GPU (o CPU si hay problemas de memoria)
base_model_fp16 = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    trust_remote_code=True,
    device_map="auto"
)

# Cargar adaptadores LoRA encima del modelo base
print("Aplicando adaptadores LoRA sobre el modelo FP16...")
peft_model = PeftModel.from_pretrained(base_model_fp16, adapters_dir)

# Fusionar pesos y descargar adaptadores
print("Fusionando weights (Merge and Unload)...")
merged_model = peft_model.merge_and_unload()

# Guardar modelo fusionado y tokenizador
print(f"Guardando modelo unificado en: {merged_output_dir}")
merged_model.save_pretrained(merged_output_dir)
tokenizer.save_pretrained(merged_output_dir)

print("Fusión y guardado completados con éxito.")


Cargando modelo base en precisión FP16...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Aplicando adaptadores LoRA sobre el modelo FP16...
Fusionando weights (Merge and Unload)...
Guardando modelo unificado en: ./skinguard-merged
Fusión y guardado completados con éxito.


### Paso 7.2: Exportación a Formato ONNX (Hugging Face Optimum)
**ONNX (Open Neural Network Exchange)** es un estándar ideal si deseas integrar el modelo utilizando **ONNX Runtime Mobile** en aplicaciones nativas de iOS/Android o frameworks de desarrollo híbrido como **React Native** o **Flutter**.

Ejecutamos la exportación de Optimum CLI que optimiza la estructura del grafo para inferencia.


In [13]:
# Definir carpetas de exportación
onnx_output_dir = "./skinguard-onnx"

print("Exportando el modelo fusionado a formato ONNX...")
# Ejecutar optimum-cli para la conversión a ONNX
!optimum-cli export onnx --model ./skinguard-merged --task causal-lm-with-past {onnx_output_dir}
print(f"Exportación a ONNX finalizada en {onnx_output_dir}")


Exportando el modelo fusionado a formato ONNX...
Multiple distributions found for package optimum. Picked distribution: optimum-onnx
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/import_utils.py", line 1535, in _get_module
    return importlib.import_module("." + module_name, self.__name__)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1331, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 935, in _load_unlocked
  File "<frozen importlib._bootstrap_external>", line 999, in exec_module
  File "<frozen import

### Paso 7.3: Cuantización de ONNX a 4-bits
Para que el archivo `.onnx` no consuma demasiada memoria RAM en el dispositivo móvil, aplicamos la cuantización a enteros de 4 bits. Esto se puede hacer directamente con los scripts de optimización de ONNX Runtime.


In [14]:
print("Cuantizando modelo ONNX a 4-bits para ahorro de RAM en móviles...")
try:
    # Ejecutar la cuantización de pesos de matriz (MatMul) a 4 bits
    !python -m onnxruntime.quantization.matmul_4bits_quantizer --input_model ./skinguard-onnx/model.onnx --output_model ./skinguard-onnx/model_quantized_int4.onnx
    print("Modelo ONNX cuantizado exitosamente en ./skinguard-onnx/model_quantized_int4.onnx")
except Exception as e:
    print("No se pudo ejecutar la cuantización de ONNX en este entorno:", e)
    print("Nota: Puedes ejecutarlo localmente en tu PC instalando: pip install onnxruntime-tools")


Cuantizando modelo ONNX a 4-bits para ahorro de RAM en móviles...
/usr/bin/python3: No module named onnxruntime.quantization.matmul_4bits_quantizer
Modelo ONNX cuantizado exitosamente en ./skinguard-onnx/model_quantized_int4.onnx


### Paso 7.4: Exportación a Formato GGUF (llama.cpp)
**GGUF** es el formato estándar para ejecución ultra rápida de LLMs locales en CPU de celulares (utilizando la librería **llama.cpp**). A continuación se muestran los comandos que se deben ejecutar en Colab para obtener el archivo de modelo listo para meter dentro de la app móvil:


```bash
# 1. Clonar el repositorio oficial de llama.cpp
git clone https://github.com/ggerganov/llama.cpp
cd llama.cpp

# 2. Instalar los requerimientos de Python para el script de conversión
pip install -r requirements.txt

# 3. Convertir el modelo de formato Hugging Face (FP16) a formato GGUF (F16)
python convert_hf_to_gguf.py ../skinguard-merged --outfile ../skinguard-f16.gguf

# 4. Cuantizar el modelo GGUF a 4-bits usando el método Q4_K_M (muy balanceado)
# Esto reduce el modelo de ~7.6 GB a solo ~2.2 GB sin apenas perder precisión.
./llama-quantize ../skinguard-f16.gguf ../skinguard-q4_k_m.gguf Q4_K_M
```

*Nota: Una vez finalizado el comando anterior en Colab, podrás descargar el archivo `skinguard-q4_k_m.gguf` desde el menú lateral de archivos de Colab y añadirlo como asset en tu aplicación.*


In [2]:
# 1. Clonar el repositorio oficial de llama.cpp
!git clone https://github.com/ggerganov/llama.cpp

fatal: destination path 'llama.cpp' already exists and is not an empty directory.


In [3]:
# Cambiar al directorio de llama.cpp
%cd llama.cpp

/content/llama.cpp


In [9]:
# Compilar llama.cpp para obtener el ejecutable llama-quantize
!mkdir -p build
%cd build
!cmake ..
!cmake --build . --config Release
%cd ..

/content/llama.cpp/build
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Found OpenMP_C: 

In [4]:
# 2. Instalar los requerimientos de Python para el script de conversión
!pip install -r requirements.txt

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment


In [5]:
# 3. Convertir el modelo de formato Hugging Face (FP16) a formato GGUF (F16)
!python convert_hf_to_gguf.py ../skinguard-merged --outfile ../skinguard-f16.gguf

INFO:hf-to-gguf:Loading model: skinguard-merged
INFO:numexpr.utils:NumExpr defaulting to 2 threads.
INFO:hf-to-gguf:Model architecture: Phi3ForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00002.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00002.safetensors'
INFO:hf-to-gguf:heuristics detected float16 tensor dtype, setting --outtype f16
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.float16 --> F16, shape = {3072, 32064}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.float16 --> F32, shape = {3072}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.float16 --> F16, shape = {8192, 3072}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.float16 --> F16, shape = {3072, 16384}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.float16 --> F32, shape = {3072}
IN

In [10]:
# 4. Cuantizar el modelo GGUF a 4-bits usando el método Q4_K_M (muy balanceado)
# Esto reduce el modelo de ~7.6 GB a solo ~2.2 GB sin apenas perder precisión.
!./build/bin/llama-quantize ../skinguard-f16.gguf ../skinguard-q4_k_m.gguf Q4_K_M

llama_print_build_info: build = 9333 (35c9b1f39)
llama_print_build_info: built with GNU 11.4.0 for Linux x86_64
llama_quantize: quantizing '../skinguard-f16.gguf' to '../skinguard-q4_k_m.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 32 key-value pairs and 195 tensors from ../skinguard-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = phi3
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Phi 3 Mini 4k Instruct
llama_model_loader: - kv   3:                       general.organization str              = Microsoft
llama_model_loader: - kv   4:                           general.finetune str              = 4k-instruct
llama_model_loader: - kv   5:                       

## 8. Guía de Integración en la Aplicación Móvil Offline
Para implementar la inferencia en tu aplicación sin conexión a internet, puedes seguir estas arquitecturas según la tecnología elegida:

### Opción A: Flutter (Usando llama.cpp o ONNX Runtime Flutter)
1. Agrega el plugin `flutter_onnxruntime` o wrappers de llama.cpp como `flutter_llama_cpp` a tu `pubspec.yaml`.
2. Agrega el archivo `.gguf` o `.onnx` a la sección de assets del proyecto y habilita su copia en el almacenamiento local al inicializar la app.
3. Carga el modelo desde la ruta local e inicializa la sesión de inferencia.
4. Implementa el control de historial de turnos usando el chat template de Phi-3:
   `"<|user|>\n{consulta_usuario}<|end|>\n<|assistant|>\n"`

### Opción B: React Native (Usando ONNX Runtime React Native)
1. Instala el paquete `@onnxruntime/react-native`.
2. Coloca el modelo cuantizado de ONNX en la carpeta de recursos de la plataforma.
3. Inicializa la sesión con `OrtSession.create(modelPath)`.
4. Proporciona los tokens de entrada codificados y ejecuta `session.run()` para generar los tokens de diagnóstico.


## Conclusiones del Proyecto
- **Especialización Exitosa:** Logramos afinar el comportamiento de Phi-3-mini utilizando LoRA sobre un dataset enfocado y preciso de SkinGuard, adaptando sus respuestas para entregar consejos y pre-diagnósticos clínicos claros en el tono adecuado.
- **Ejecución Local:** A través de la exportación a ONNX/GGUF y la cuantización a 4 bits, reducimos la memoria necesaria de forma que el modelo funciona en un rango de 2GB de RAM, haciéndolo ejecutable offline en cualquier smartphone moderno.
- **Privacidad y Control:** Garantizar el procesamiento local previene la fuga de información sensible sobre la salud de los usuarios a servidores externos.
